# Notebook 3: Monte Carlo 실험 & DOE (v0.4)
## 한국군 방공체계 M&S — 반복실험 및 통계 분석

이 노트북에서는:
- 시나리오(7) × 아키텍처(2) × 반복(300회) 매트릭스 실행
- 12개 성능 메트릭 자동 수집 (v0.4: 다중 교전 메트릭 2개 추가)
- 결과 저장 및 체크포인팅

In [ ]:
import sys, os, time, json
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install simpy mesa networkx matplotlib pandas numpy scipy seaborn --quiet
    from google.colab import drive
    drive.mount('/content/drive')
    sys.path.insert(0, '/content/drive/MyDrive/K-ADS_Simulation')
else:
    sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from modules.config import EXPERIMENT_CONFIG, SCENARIO_PARAMS
from modules.model import AirDefenseModel

# 결과 저장 경로
RESULTS_DIR = 'results'
os.makedirs(RESULTS_DIR, exist_ok=True)
print('모듈 로드 완료!')

---
## 실험 설정

In [ ]:
# 실험 파라미터
N_RUNS = EXPERIMENT_CONFIG['monte_carlo_runs']  # 300회
ARCHITECTURES = EXPERIMENT_CONFIG['architectures']  # ['linear', 'killweb']
SCENARIOS = EXPERIMENT_CONFIG['scenarios']  # ['scenario_1_saturation', 'scenario_5_node_destruction']
CHECKPOINT_INTERVAL = EXPERIMENT_CONFIG['checkpoint_interval']  # 50회

print(f'반복 횟수: {N_RUNS}')
print(f'아키텍처: {ARCHITECTURES}')
print(f'시나리오: {SCENARIOS}')
print(f'총 실행 수: {N_RUNS * len(ARCHITECTURES) * len(SCENARIOS)} = '
      f'{N_RUNS} × {len(ARCHITECTURES)} × {len(SCENARIOS)}')

---
## 소규모 파일럿 (10회)

In [ ]:
# 파일럿 실행 (10회) — 시간 추정 및 통계 분포 확인
PILOT_RUNS = EXPERIMENT_CONFIG['pilot_runs']  # 10
pilot_results = []

start_time = time.time()
for scenario in SCENARIOS:
    for arch in ARCHITECTURES:
        for run in range(PILOT_RUNS):
            model = AirDefenseModel(
                architecture=arch,
                scenario=scenario,
                seed=run * 1000 + hash(arch) % 1000
            )
            result = model.run_full()
            flat = result['metrics_flat']
            flat['architecture'] = arch
            flat['scenario'] = scenario
            flat['run'] = run
            pilot_results.append(flat)

pilot_time = time.time() - start_time
df_pilot = pd.DataFrame(pilot_results)

print(f'파일럿 완료: {len(pilot_results)}회 / {pilot_time:.1f}초')
print(f'예상 전체 실행 시간: {pilot_time / len(pilot_results) * N_RUNS * len(ARCHITECTURES) * len(SCENARIOS):.0f}초')
print(f'\n--- 파일럿 결과 요약 ---')
print(df_pilot.groupby(['scenario', 'architecture'])[['leaker_rate', 'engagement_success_rate',
    'sensor_to_shooter_time_mean']].describe().round(2))

---
## 본 실험 (300회 반복)

In [ ]:
def run_experiment(n_runs, architectures, scenarios, checkpoint_interval=50,
                   results_dir='results'):
    """Monte Carlo 반복실험 실행 (체크포인팅 포함)"""
    all_results = []
    total_runs = n_runs * len(architectures) * len(scenarios)
    
    # 기존 체크포인트 로드
    checkpoint_file = os.path.join(results_dir, 'checkpoint.csv')
    start_from = 0
    if os.path.exists(checkpoint_file):
        df_existing = pd.read_csv(checkpoint_file)
        all_results = df_existing.to_dict('records')
        start_from = len(all_results)
        print(f'체크포인트 로드: {start_from}개 기존 결과')
    
    run_idx = 0
    pbar = tqdm(total=total_runs, initial=start_from, desc='실험 진행')
    
    for scenario in scenarios:
        for arch in architectures:
            for run in range(n_runs):
                run_idx += 1
                if run_idx <= start_from:
                    continue
                
                seed = run * 7919 + hash(f'{arch}_{scenario}') % 10000
                model = AirDefenseModel(
                    architecture=arch,
                    scenario=scenario,
                    seed=seed
                )
                result = model.run_full()
                flat = result['metrics_flat']
                flat['architecture'] = arch
                flat['scenario'] = scenario
                flat['run'] = run
                flat['seed'] = seed
                all_results.append(flat)
                pbar.update(1)
                
                # 체크포인팅
                if len(all_results) % checkpoint_interval == 0:
                    df = pd.DataFrame(all_results)
                    df.to_csv(checkpoint_file, index=False)
                    pbar.set_postfix({'saved': len(all_results)})
    
    pbar.close()
    
    # 최종 저장
    df = pd.DataFrame(all_results)
    final_file = os.path.join(results_dir, 'experiment_results.csv')
    df.to_csv(final_file, index=False)
    
    # 체크포인트 파일 정리
    if os.path.exists(checkpoint_file):
        os.remove(checkpoint_file)
    
    print(f'\n실험 완료! {len(all_results)}개 결과 저장 → {final_file}')
    return df

# 본 실험 실행
df_results = run_experiment(
    n_runs=N_RUNS,
    architectures=ARCHITECTURES,
    scenarios=SCENARIOS,
    checkpoint_interval=CHECKPOINT_INTERVAL,
    results_dir=RESULTS_DIR
)

In [ ]:
# 결과 요약
print('=== 실험 결과 요약 ===')
summary_metrics = [
    'leaker_rate', 'engagement_success_rate',
    'sensor_to_shooter_time_mean', 'ammo_efficiency',
    'max_concurrent_engagements', 'c2_throughput',
    'multi_engagement_rate', 'avg_shooters_per_multi_engagement',
]
available = [m for m in summary_metrics if m in df_results.columns]
summary = df_results.groupby(['scenario', 'architecture'])[available].agg(
    ['mean', 'std']
).round(2)

print(summary)
summary.to_csv(os.path.join(RESULTS_DIR, 'experiment_summary.csv'))
print(f'\n요약 저장 → {RESULTS_DIR}/experiment_summary.csv')

In [ ]:
# 수렴성 검사 — 반복 횟수 증가에 따른 평균 안정화 확인
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
metrics_to_check = ['leaker_rate', 'engagement_success_rate',
                     'sensor_to_shooter_time_mean', 'ammo_efficiency']
metric_labels = ['누출률 (%)', '교전 성공률 (%)', '센서-투-슈터 시간 (s)', '탄약 효율']

for idx, (metric, label) in enumerate(zip(metrics_to_check, metric_labels)):
    ax = axes[idx // 2][idx % 2]
    for scenario in SCENARIOS:
        for arch in ARCHITECTURES:
            subset = df_results[
                (df_results['scenario'] == scenario) &
                (df_results['architecture'] == arch)
            ][metric].values
            
            cumulative_mean = np.cumsum(subset) / np.arange(1, len(subset) + 1)
            s_name = scenario.replace('scenario_', 'S').split('_')[0]
            ax.plot(cumulative_mean, label=f'{s_name}-{arch}', linewidth=1.5)
    
    ax.set_xlabel('반복 횟수')
    ax.set_ylabel(label)
    ax.set_title(f'수렴성 검사: {label}')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

fig.suptitle('Monte Carlo 수렴성 검사', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'convergence_check.png'), dpi=150, bbox_inches='tight')
plt.show()